In [15]:
import os

In [16]:
%pwd

'c:\\Users\\Raghav\\Desktop\\Image_Captioning_using_DeepLearning\\research'

In [17]:
# Changing working directory one path back

os.chdir("../")

In [18]:
%pwd

'c:\\Users\\Raghav\\Desktop\\Image_Captioning_using_DeepLearning'

In [19]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path           
    updated_base_model_path: Path   

    IMAGE_SIZE: list         
    WEIGHTS: str            
    INCLUDE_TOP: bool        
    POOLING: str

    CNN_DIM: int
    VOCAB_SIZE: int          
    MAX_LENGTH: int          
    EMBEDDING_DIM: int       
    LSTM_UNITS: int              
    DROPOUT: float          
    LEARNING_RATE: float

    CLIPNORM : float

In [20]:
from imgCaption.constants import *
from imgCaption.utils.common import read_yaml,create_directories

In [21]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        params = self.params.prepare_base_model_params

        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
                                                root_dir=config.root_dir,
                                                base_model_path=config.base_model_path,
                                                updated_base_model_path=config.updated_base_model_path,
                                                IMAGE_SIZE=params.IMAGE_SIZE,
                                                WEIGHTS=params.WEIGHTS,
                                                INCLUDE_TOP=params.INCLUDE_TOP,
                                                POOLING=params.POOLING,
                                                CNN_DIM=params.CNN_DIM,
                                                VOCAB_SIZE=params.VOCAB_SIZE,
                                                MAX_LENGTH=params.MAX_LENGTH,
                                                EMBEDDING_DIM=params.EMBEDDING_DIM,
                                                LSTM_UNITS=params.LSTM_UNITS,
                                                DROPOUT=params.DROPOUT,
                                                LEARNING_RATE=params.LEARNING_RATE,
                                                CLIPNORM=params.CLIPNORM
                                            )

        return prepare_base_model_config

In [22]:
import tensorflow as tf
from tensorflow.keras.layers import Input,Dense,LSTM,Embedding,Dropout,Concatenate
                            
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from imgCaption import logger

In [23]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def get_base_model(self):
        self.model = tf.keras.applications.resnet50.ResNet50(
                                input_shape=self.config.IMAGE_SIZE,  
                                weights=self.config.WEIGHTS,                 
                                include_top=self.config.INCLUDE_TOP,         
                                pooling=self.config.POOLING                  
                                                            )

        self.model.trainable = False

        self.save_model(path=self.config.base_model_path, model=self.model)

    @staticmethod
    def prepare_full_model(cnn_dim, vocab_size,max_caption_length,embedding_dim,
                            lstm_units,dropout_rate,learning_rate,clipnorm):
        
        input_img_features = Input(shape=(cnn_dim,), name = "Img_Features_Input")
        img_drop_layer = Dropout(dropout_rate, name="img_drop_layer")(input_img_features)
        img_output_tensor = Dense(embedding_dim, activation='relu', name="img_output_tensor")(img_drop_layer)
        
        input_caption = Input(shape=(max_caption_length,), name="Caption_Input")
        cap_embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name="cap_embedding_layer")(input_caption)
        cap_drop_layer = Dropout(dropout_rate, name="cap_drop_layer")(cap_embedding_layer)
        cap_output_tensor = LSTM(lstm_units, name="cap_output_tensor_LSTM")(cap_drop_layer)

        merged_tensor = Concatenate(axis=-1)([img_output_tensor,cap_output_tensor])

        merged_layer = Dense(embedding_dim, activation='relu', name="merged_layer")(merged_tensor)
        merged_drop_layer = Dropout(dropout_rate, name="merged_drop_layer")(merged_layer)
        output_layer = Dense(vocab_size, activation='softmax', name="Output_Layer_Final")(merged_drop_layer)

        full_model = Model(
            inputs=[input_img_features, input_caption],
            outputs=output_layer,
            name='Image_Captioning'
        )

        full_model.compile(
            optimizer=Adam(learning_rate=learning_rate, clipnorm=clipnorm),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        full_model.summary()
        return full_model

    def update_base_model(self):

        self.full_model = self.prepare_full_model(
                            cnn_dim=self.config.CNN_DIM,                               
                            vocab_size=self.config.VOCAB_SIZE,
                            max_caption_length=self.config.MAX_LENGTH,
                            embedding_dim=self.config.EMBEDDING_DIM,
                            lstm_units=self.config.LSTM_UNITS,
                            dropout_rate=self.config.DROPOUT,
                            learning_rate=self.config.LEARNING_RATE,
                            clipnorm=self.config.CLIPNORM
                            )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)
        logger.info(f"Full caption model saved at: {self.config.updated_base_model_path}")

In [24]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-06-27 12:28:11,937: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-27 12:28:11,976: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-27 12:28:11,981: INFO: common: created directory at: artifacts]
[2026-06-27 12:28:11,987: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-06-27 12:28:16,258: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Model: "Image_Captioning"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 Caption_Input (InputLayer)     [(None, 35)]         0           []                               
                                                                                                  
 Img_Features_Input (InputLayer  [(No

A "Keras tensor" is a symbolic tensor, such as a tensor that was created via Input(). A "symbolic tensor" can be understood as a placeholder – it does not contain any actual numerical data, only a shape and dtype. It can be used for building Functional models, but it cannot be used in actual computations.

Keras Tensor is a blueprint/map of your model — it defines what shape data will be and how it flows through layers, before any real data arrives.